# BLOQUE 01 – PERFIL DEMOGRÁFICO
## Objetivo
Construir capa consolidada demográfica normalizada por unidad espacial.
## Inputs
- poblacion.shp
- genero.shp
- discapacidad.csv
## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output
-D5C4V1 Mapas de Riesgo
- 1 Áreas con altos índices de inseguridad
- 2 Zona inundable


In [7]:
import geopandas as gpd
import pandas as pd
import os

In [8]:
# --- 1. CONFIGURACIÓN DE RUTAS ---
Delitos = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\Delitos\Delitos Jalisco.gpkg"
Inundaciones = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\Inundación\Inundación.gpkg"
carpeta_salida = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\D5C4V1 Mapas de Riesgo"
nombre_archivo_final = "1_D5C4V1_output_Mapas_de_Riesgo"

In [9]:
if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)

In [11]:
# --- 2. CARGA DE DATOS ---
print("Cargando capas...")
try:
    # Leemos los archivos
    gdf_delitos = gpd.read_file(ruta_delitos)
    gdf_inundaciones = gpd.read_file(ruta_inundaciones)
    print("✅ Archivos leídos correctamente.")

    # --- 3. DETECTOR DE COLUMNAS (Dentro del try para asegurar que existan los GDF) ---
    def buscar_columna(gdf, nombre_buscado):
        columnas = gdf.columns.tolist()
        for c in columnas:
            if nombre_buscado.lower() in c.lower():
                return c
        return None

    col_vulnerabil = buscar_columna(gdf_inundaciones, 'Vulnerabil')
    col_delito = buscar_columna(gdf_delitos, 'delito')

    if not col_vulnerabil or not col_delito:
        print(f"⚠️ Columnas no encontradas. Disponibles en Inundación: {gdf_inundaciones.columns.tolist()}")
    else:
        # --- 4. PROCESAMIENTO ---
        print("Homologando datos...")
        
        # Crear la columna común
        gdf_delitos['Categoría'] = gdf_delitos[col_delito].astype(str)
        gdf_inundaciones['Categoría'] = gdf_inundaciones[col_vulnerabil].astype(str)

        # Alinear coordenadas
        if gdf_delitos.crs != gdf_inundaciones.crs:
            gdf_inundaciones = gdf_inundaciones.to_crs(gdf_delitos.crs)

        # Unir capas
        columnas_finales = ['Categoría', 'geometry']
        gdf_final = pd.concat([gdf_delitos[columnas_finales], gdf_inundaciones[columnas_finales]], ignore_index=True)
        
        # Convertir a GeoDataFrame oficial
        gdf_final = gpd.GeoDataFrame(gdf_final, geometry='geometry', crs=gdf_delitos.crs)

        # --- 5. GUARDADO ---
        ruta_final = os.path.join(carpeta_salida, nombre_archivo_final)
        gdf_final.to_file(ruta_final, driver="GPKG")

        print(f"\n✅ ¡ÉXITO!")
        print(f"📍 Guardado en: {ruta_final}")
        print(gdf_final['Categoría'].head())

except FileNotFoundError:
    print("❌ Error: No se encontraron los archivos en las rutas especificadas. Revisa las carpetas.")
except Exception as e:
    print(f"❌ Ocurrió un error inesperado: {e}")

Cargando capas...
❌ Ocurrió un error inesperado: name 'ruta_delitos' is not defined
